In [3]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

BACKEND_DIR = str(Path(os.getcwd()).resolve().parent) if os.path.basename(os.getcwd()) == "monitor" else str(Path(os.getcwd()).resolve() / "backend")
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

HF_TOKEN = os.environ.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score
import torch

with open("../datasets/Sentences_AllAgree.txt", "r", encoding="latin-1") as f:
    lines = f.readlines()

sentences, labels = [], []
label_map = {"positive": 0, "negative": 1, "neutral": 2}

for line in lines:
    line = line.strip()
    if "@" in line:
        sentence, label = line.rsplit("@", 1)
        sentences.append(sentence.strip())
        labels.append(label_map[label.strip()])

df = pd.DataFrame({"sentence": sentences, "labels": labels})
print(f"Loaded {len(df)} sentences")
print(df["labels"].value_counts())

train_df, eval_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["labels"]
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))

KeyError: 'positive'

In [ ]:
# load finBERT tokenizer
model_name = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=128,
        padding=False
    )

train_dataset = train_dataset.map(tokenize_fn, batched=True)
eval_dataset = eval_dataset.map(tokenize_fn, batched=True)

# Remove raw sentence column
train_dataset = train_dataset.remove_columns(["sentence"])
eval_dataset = eval_dataset.remove_columns(["sentence"])

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, ignore_mismatched_sizes=True)


# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return { 
        "accuracy": accuracy_score(labels, predictions),
        "f1":f1_score(labels, predictions, average="weighted")
    }


# Training Arguments
training_args = TrainingArguments(
    output_dir="./finbert-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

print("Starting training...")
train_result = trainer.train()

# Get final eval metrics from training log history
eval_metrics = [m for m in trainer.state.log_history if "eval_accuracy" in m]
if eval_metrics:
    last = eval_metrics[-1]
    print(f"\nFinal Accuracy: {last['eval_accuracy']:.4f}")
    print(f"Final F1:       {last['eval_f1']:.4f}")

# Save the fine-tuned model and tokenizer
trainer.save_model("./finbert-finetuned")
tokenizer.save_pretrained("./finbert-finetuned")
print("Model saved to ./finbert-finetuned")


Map:   0%|          | 0/1811 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Train: 1811 | Eval: 453


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Final Accuracy: 0.9868
Final F1:       0.9868


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./finbert-finetuned


In [ ]:

# Test

from transformers import pipeline
sentiment = pipeline(
    "text-classification",
    model="./finbert-finetuned",
    tokenizer="./finbert-finetuned"
)

test_sentences = [
    "Apple stock surged after strong earnings beat expectations",
    "The company reported a significant decline in quarterly revenue",
    "Trading volume remained steady throughout the session"
]

for s in test_sentences:
    result = sentiment(s)[0]
    print(f"{s}\n→ {result['label']} ({result['score']:.3f})\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Apple stock surged after strong earnings beat expectations
→ neutral (0.989)

The company reported a significant decline in quarterly revenue
→ positive (0.984)

Trading volume remained steady throughout the session
→ negative (0.533)



In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo("umenyioraj/finbert-financial-sentiment", exist_ok=True)
api.upload_folder(
    folder_path="./finbert-finetuned",
    repo_id="umenyioraj/finbert-financial-sentiment",
)
print("Model pushed!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model pushed!
